In [1]:
# ============================================================
# 08B_AURORA_aligned_oos_reanalysis.ipynb
# AURORA-TWETF Aligned OOS Reanalysis
#
# Purpose:
# 1. Reuse Notebook 08 signal weights.
# 2. Force AURORA and benchmark policies to use the exact same OOS dates.
# 3. Force strict test-only comparisons to use the exact same test dates.
# 4. Recompute performance, rankings, and AURORA-vs-benchmark tables.
# 5. Remove the 508-day vs 545-day date-count mismatch from Notebook 08.
#
# Important:
# - Educational/research backtest only.
# - Not personalized financial advice.
# - This notebook does not create new models.
# - It only re-evaluates existing Notebook 08 policies on aligned dates.
# ============================================================

from __future__ import annotations

import json
import math
import hashlib
import warnings
from pathlib import Path
from datetime import datetime, timezone

warnings.filterwarnings("ignore")

# ============================================================
# 0. Colab setup
# ============================================================

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    print("Google Drive mount skipped or failed.")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================================
# 1. Paths and configuration
# ============================================================

PROJECT_CODE = "AURORA_TWETF"
PUBLICATION_ROOT = Path("/content/drive/MyDrive/AURORA_TWETF")

DATA_ROOT = PUBLICATION_ROOT / "data"
PANEL_DIR = DATA_ROOT / "panels"
MODELING_DIR = DATA_ROOT / "modeling"

OUTPUT_ROOT = PUBLICATION_ROOT / "outputs" / PROJECT_CODE
TABLE_DIR = OUTPUT_ROOT / "tables"
REPORT_DIR = OUTPUT_ROOT / "reports"
FIGURE_DIR = OUTPUT_ROOT / "figures"

# Notebook 07 and Notebook 08 run IDs.
NOTEBOOK07_RUN_ID = "20260624_031817"
NOTEBOOK08_RUN_ID = "20260624_034204"

NOTEBOOK07_ROOT = OUTPUT_ROOT / "purged_walk_forward_models" / f"run_{NOTEBOOK07_RUN_ID}"
NOTEBOOK08_ROOT = OUTPUT_ROOT / "stronger_financial_baselines_purged_allocation" / f"run_{NOTEBOOK08_RUN_ID}"

NOTEBOOK08_INPUT_INDEX = NOTEBOOK07_ROOT / "NOTEBOOK08_OR_ALLOCATION_INPUT_INDEX_PURGED_WF.csv"

NOTEBOOK08_WEIGHT_DIR = NOTEBOOK08_ROOT / "weights"
NOTEBOOK08_TABLE_DIR = NOTEBOOK08_ROOT / "tables"

RUN_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

RUN_ROOT = OUTPUT_ROOT / "aligned_oos_reanalysis" / f"run_{RUN_ID}"

WEIGHT_DIR = RUN_ROOT / "weights"
RETURN_DIR = RUN_ROOT / "returns"
PLOT_DIR = RUN_ROOT / "plots"
TABLE_RUN_DIR = RUN_ROOT / "tables"
REPORT_RUN_DIR = RUN_ROOT / "reports"
PAPER_FIGURE_DIR = RUN_ROOT / "paper_figures"
DIAGNOSTIC_DIR = RUN_ROOT / "diagnostics"

for d in [
    OUTPUT_ROOT,
    TABLE_DIR,
    REPORT_DIR,
    FIGURE_DIR,
    RUN_ROOT,
    WEIGHT_DIR,
    RETURN_DIR,
    PLOT_DIR,
    TABLE_RUN_DIR,
    REPORT_RUN_DIR,
    PAPER_FIGURE_DIR,
    DIAGNOSTIC_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print("AURORA-TWETF Notebook 08B: Aligned OOS Reanalysis")
print("=" * 80)
print("Timestamp UTC       :", RUN_TIMESTAMP)
print("Run ID              :", RUN_ID)
print("Notebook 07 root    :", NOTEBOOK07_ROOT)
print("Notebook 08 root    :", NOTEBOOK08_ROOT)
print("Notebook 08 registry:", NOTEBOOK08_INPUT_INDEX)
print("Run root            :", RUN_ROOT)
print("=" * 80)

if not NOTEBOOK07_ROOT.exists():
    raise FileNotFoundError(f"Notebook 07 root not found: {NOTEBOOK07_ROOT}")

if not NOTEBOOK08_ROOT.exists():
    raise FileNotFoundError(f"Notebook 08 root not found: {NOTEBOOK08_ROOT}")

if not NOTEBOOK08_INPUT_INDEX.exists():
    raise FileNotFoundError(f"Notebook 08 input index not found: {NOTEBOOK08_INPUT_INDEX}")

if not NOTEBOOK08_WEIGHT_DIR.exists():
    raise FileNotFoundError(f"Notebook 08 weight directory not found: {NOTEBOOK08_WEIGHT_DIR}")

# ============================================================
# 2. Global settings
# ============================================================

ETF_UNIVERSE = ["0050", "006208", "00692", "00881"]
CASH_COL = "CASH"
ALL_ASSETS = ETF_UNIVERSE + [CASH_COL]

ANNUALIZATION_DAYS = 252
INITIAL_CAPITAL = 1.0

TRANSACTION_COST_RATE = 0.0010
REBALANCE_FREQUENCY = "monthly"

AURORA_POLICY_PREFIX = "AURORA"

BENCHMARK_TYPES = [
    "passive_benchmark",
    "dynamic_financial_baseline",
]

# ============================================================
# 3. Utility functions
# ============================================================

def save_json(path, obj):
    Path(path).write_text(
        json.dumps(obj, indent=2, ensure_ascii=False, default=str),
        encoding="utf-8",
    )

def sha256_file(path, chunk_size=1024 * 1024):
    path = Path(path)
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

def make_file_manifest(root):
    root = Path(root)
    rows = []

    for p in sorted(root.rglob("*")):
        if p.is_file():
            stat = p.stat()
            rows.append({
                "path": p.relative_to(root).as_posix(),
                "size_bytes": int(stat.st_size),
                "modified_utc": datetime.fromtimestamp(
                    stat.st_mtime,
                    timezone.utc,
                ).strftime("%Y-%m-%dT%H:%M:%SZ"),
                "sha256": sha256_file(p),
            })

    return pd.DataFrame(rows)

def safe_name(x):
    return (
        str(x)
        .replace("/", "_")
        .replace("\\", "_")
        .replace(":", "_")
        .replace(" ", "_")
        .replace(".", "_")
    )

def clean_symbol_name(x):
    x = str(x)
    x = x.replace(".TW", "")
    x = x.replace(".TWO", "")
    x = x.replace("TW_", "")
    return x

def find_first_existing(paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    return None

def read_table_auto(path):
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")

    if path.suffix.lower() == ".parquet":
        df = pd.read_parquet(path)
    elif path.suffix.lower() == ".csv":
        df = pd.read_csv(path)
    else:
        raise ValueError(f"Unsupported file type: {path}")

    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"])
        df = df.set_index("date")
    else:
        try:
            df.index = pd.to_datetime(df.index)
        except Exception:
            pass

    df.index.name = "date"
    return df.sort_index()

def load_etf_return_panel():
    candidates = [
        PANEL_DIR / "AURORA_etf_return_panel.parquet",
        PANEL_DIR / "AURORA_etf_returns_panel.parquet",
        PANEL_DIR / "AURORA_return_panel.parquet",
        MODELING_DIR / "AURORA_etf_return_panel.parquet",
    ]

    path = find_first_existing(candidates)

    if path is None:
        raise FileNotFoundError(
            "Could not find ETF return panel. Tried:\n"
            + "\n".join(str(p) for p in candidates)
        )

    df = pd.read_parquet(path)
    df.index = pd.to_datetime(df.index)
    df.index.name = "date"
    df = df.sort_index()
    df = df.rename(columns={c: clean_symbol_name(c) for c in df.columns})

    missing = [s for s in ETF_UNIVERSE if s not in df.columns]
    if missing:
        raise ValueError(
            f"ETF return panel found at {path}, but missing ETF columns: {missing}\n"
            f"Available columns: {list(df.columns)}"
        )

    df = df[ETF_UNIVERSE].copy()
    df = df.replace([np.inf, -np.inf], np.nan).fillna(0.0)

    return df, path

def normalize_rows(df):
    out = df.copy()
    out = out.reindex(columns=ALL_ASSETS).fillna(0.0)
    out = out.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    out[out < 0] = 0.0

    row_sums = out.sum(axis=1)
    zero_mask = row_sums <= 0

    if zero_mask.any():
        out.loc[zero_mask, ETF_UNIVERSE] = 1.0 / len(ETF_UNIVERSE)
        out.loc[zero_mask, CASH_COL] = 0.0
        row_sums = out.sum(axis=1)

    out = out.div(row_sums, axis=0)
    return out

def get_rebalance_dates(index, frequency):
    idx = pd.DatetimeIndex(index).sort_values()

    if frequency == "monthly":
        groups = pd.Series(idx, index=idx).groupby([idx.year, idx.month])
    elif frequency == "quarterly":
        groups = pd.Series(idx, index=idx).groupby([idx.year, idx.quarter])
    elif frequency == "weekly":
        iso = idx.isocalendar()
        groups = pd.Series(idx, index=idx).groupby([iso.year, iso.week])
    else:
        raise ValueError(f"Unsupported rebalance frequency: {frequency}")

    dates = []
    for _, values in groups:
        dates.append(values.iloc[0])

    return pd.DatetimeIndex(dates)

def expand_rebalance_weights_to_daily(signal_weight_df, daily_index, rebalance_dates):
    cols = ALL_ASSETS
    signal = signal_weight_df.reindex(columns=cols).fillna(0.0).copy()
    signal = normalize_rows(signal)
    signal_idx = pd.DatetimeIndex(signal.index).sort_values()

    daily = pd.DataFrame(index=daily_index, columns=cols, dtype=float)

    for i, reb_date in enumerate(rebalance_dates):
        if i + 1 < len(rebalance_dates):
            period_idx = daily_index[(daily_index >= reb_date) & (daily_index < rebalance_dates[i + 1])]
        else:
            period_idx = daily_index[daily_index >= reb_date]

        prior_signals = signal_idx[signal_idx < reb_date]

        if len(prior_signals) == 0:
            signal_date = signal_idx[0]
        else:
            signal_date = prior_signals[-1]

        daily.loc[period_idx, cols] = signal.loc[signal_date, cols].values

    daily = daily.ffill().bfill()
    daily = normalize_rows(daily)

    return daily

def compute_turnover(daily_weights, rebalance_dates):
    turnover = pd.Series(0.0, index=daily_weights.index)
    prev_w = None

    for dt in rebalance_dates:
        if dt not in daily_weights.index:
            continue

        w = daily_weights.loc[dt, ALL_ASSETS]

        if prev_w is None:
            turnover.loc[dt] = w.drop(labels=[CASH_COL], errors="ignore").abs().sum()
        else:
            turnover.loc[dt] = (w - prev_w).abs().sum() / 2.0

        prev_w = w

    return turnover

def backtest_on_fixed_index(policy_name, signal_weights, etf_returns, evaluation_index):
    """
    Backtest a policy on a fixed date index shared by every policy.
    This is the core alignment step of Notebook 08B.
    """
    evaluation_index = pd.DatetimeIndex(evaluation_index).sort_values()

    returns = etf_returns.copy()
    returns[CASH_COL] = 0.0
    returns = returns.reindex(evaluation_index)

    signal = signal_weights.copy()
    signal.index = pd.to_datetime(signal.index)
    signal = signal.sort_index()
    signal = signal.reindex(columns=ALL_ASSETS).fillna(0.0)

    # Align signal to the exact evaluation index.
    # Use only known previous weights when a date is missing.
    signal_aligned = signal.reindex(evaluation_index).ffill().bfill()
    signal_aligned = normalize_rows(signal_aligned)

    if returns[ALL_ASSETS].isna().any().any():
        missing_rows = returns[returns[ALL_ASSETS].isna().any(axis=1)]
        raise ValueError(
            f"Return panel contains missing values for policy {policy_name}. "
            f"Example missing dates: {missing_rows.index[:5].tolist()}"
        )

    rebalance_dates = get_rebalance_dates(evaluation_index, REBALANCE_FREQUENCY)

    daily_weights = expand_rebalance_weights_to_daily(
        signal_weight_df=signal_aligned,
        daily_index=evaluation_index,
        rebalance_dates=rebalance_dates,
    )

    gross_return = (daily_weights[ALL_ASSETS] * returns[ALL_ASSETS]).sum(axis=1)

    turnover = compute_turnover(daily_weights, rebalance_dates)
    transaction_cost = turnover * TRANSACTION_COST_RATE
    net_return = gross_return - transaction_cost

    equity = (1.0 + net_return).cumprod() * INITIAL_CAPITAL

    out = pd.DataFrame(index=evaluation_index)
    out.index.name = "date"
    out["policy_name"] = policy_name
    out["gross_return"] = gross_return
    out["turnover"] = turnover
    out["transaction_cost"] = transaction_cost
    out["net_return"] = net_return
    out["equity"] = equity
    out["drawdown"] = equity / equity.cummax() - 1.0
    out["is_rebalance_date"] = out.index.isin(rebalance_dates)

    return out, daily_weights

def performance_metrics(return_df):
    r = return_df["net_return"].astype(float).copy()
    equity = return_df["equity"].astype(float).copy()
    drawdown = return_df["drawdown"].astype(float).copy()

    n = len(r)
    if n == 0:
        return {}

    total_return = float(equity.iloc[-1] / equity.iloc[0] - 1.0) if equity.iloc[0] != 0 else np.nan
    annual_return = float((1.0 + total_return) ** (ANNUALIZATION_DAYS / max(n, 1)) - 1.0)

    annual_vol = float(r.std(ddof=1) * np.sqrt(ANNUALIZATION_DAYS)) if n > 1 else np.nan
    sharpe = annual_return / annual_vol if annual_vol and annual_vol > 0 else np.nan

    downside = r[r < 0]
    downside_vol = float(downside.std(ddof=1) * np.sqrt(ANNUALIZATION_DAYS)) if len(downside) > 1 else np.nan
    sortino = annual_return / downside_vol if downside_vol and downside_vol > 0 else np.nan

    max_drawdown = float(drawdown.min())
    calmar = annual_return / abs(max_drawdown) if max_drawdown < 0 else np.nan

    return {
        "n_days": int(n),
        "start_date": str(r.index.min().date()),
        "end_date": str(r.index.max().date()),
        "total_return": total_return,
        "annual_return": annual_return,
        "annual_volatility": annual_vol,
        "sharpe_ratio": sharpe,
        "sortino_ratio": sortino,
        "max_drawdown": max_drawdown,
        "calmar_ratio": calmar,
        "hit_rate": float((r > 0).mean()),
        "avg_daily_return": float(r.mean()),
        "avg_turnover": float(return_df["turnover"].mean()),
        "total_turnover": float(return_df["turnover"].sum()),
        "total_transaction_cost": float(return_df["transaction_cost"].sum()),
        "final_equity": float(equity.iloc[-1]),
    }

def add_composite_rank(df):
    out = df.copy()

    out["rank_total_return"] = out["total_return"].rank(ascending=False, method="min")
    out["rank_sharpe"] = out["sharpe_ratio"].rank(ascending=False, method="min")
    out["rank_sortino"] = out["sortino_ratio"].rank(ascending=False, method="min")
    out["rank_drawdown"] = out["max_drawdown"].rank(ascending=False, method="min")
    out["rank_calmar"] = out["calmar_ratio"].rank(ascending=False, method="min")

    out["allocation_composite_rank"] = (
        out["rank_total_return"]
        + out["rank_sharpe"]
        + out["rank_sortino"]
        + out["rank_drawdown"]
        + out["rank_calmar"]
    ) / 5.0

    out = out.sort_values(
        ["allocation_composite_rank", "sharpe_ratio", "total_return"],
        ascending=[True, False, False],
    )

    return out

def latest_fold_deduplicate(proba_df, split_filter=None):
    df = proba_df.copy()

    if split_filter is not None and "split" in df.columns:
        if isinstance(split_filter, str):
            split_filter = [split_filter]
        df = df[df["split"].isin(split_filter)].copy()

    if df.empty:
        return df

    df = df.reset_index()

    if "fold_id" not in df.columns:
        df["fold_id"] = "WF0"

    df["fold_number"] = (
        df["fold_id"]
        .astype(str)
        .str.extract(r"(\d+)", expand=False)
        .fillna("0")
        .astype(int)
    )

    split_priority = {"train": 0, "validation": 1, "test": 2}
    if "split" in df.columns:
        df["split_priority"] = df["split"].map(split_priority).fillna(0).astype(int)
    else:
        df["split_priority"] = 0

    df = df.sort_values(["date", "split_priority", "fold_number"])
    df = df.drop_duplicates(subset=["date"], keep="last")
    df = df.set_index("date").sort_index()
    df.index.name = "date"

    return df.drop(columns=["fold_number", "split_priority"], errors="ignore")

# ============================================================
# 4. Plotting functions
# ============================================================

def plot_equity_curves(all_returns_df, path, title, policies=None, top_n=None):
    if policies is None:
        final_equity = (
            all_returns_df.groupby("policy_name")["equity"]
            .last()
            .sort_values(ascending=False)
        )
        policies = final_equity.index.tolist()

    if top_n is not None:
        policies = policies[:top_n]

    plt.figure(figsize=(12, 6))
    for policy in policies:
        grp = all_returns_df[all_returns_df["policy_name"] == policy].sort_index()
        plt.plot(grp.index, grp["equity"], label=policy, linewidth=1.7)

    plt.title(title)
    plt.xlabel("Date")
    plt.ylabel("Equity, initial capital = 1")
    plt.grid(True, alpha=0.3)
    plt.legend(loc="best", fontsize=8)
    plt.tight_layout()
    plt.savefig(path, dpi=220)
    plt.close()

def plot_drawdowns(all_returns_df, path, title, policies=None, top_n=None):
    if policies is None:
        final_equity = (
            all_returns_df.groupby("policy_name")["equity"]
            .last()
            .sort_values(ascending=False)
        )
        policies = final_equity.index.tolist()

    if top_n is not None:
        policies = policies[:top_n]

    plt.figure(figsize=(12, 6))
    for policy in policies:
        grp = all_returns_df[all_returns_df["policy_name"] == policy].sort_index()
        plt.plot(grp.index, grp["drawdown"], label=policy, linewidth=1.5)

    plt.title(title)
    plt.xlabel("Date")
    plt.ylabel("Drawdown")
    plt.grid(True, alpha=0.3)
    plt.legend(loc="best", fontsize=8)
    plt.tight_layout()
    plt.savefig(path, dpi=220)
    plt.close()

def plot_metric_bar(df, metric, path, title, higher_is_better=True, top_n=None):
    tmp = df.copy()
    tmp = tmp.replace([np.inf, -np.inf], np.nan).dropna(subset=[metric])
    tmp = tmp.sort_values(metric, ascending=not higher_is_better)

    if top_n is not None:
        tmp = tmp.head(top_n)

    plt.figure(figsize=(11, max(4, 0.4 * len(tmp))))
    sns.barplot(data=tmp, y="policy_name", x=metric, color="#4C72B0")
    plt.title(title)
    plt.xlabel(metric)
    plt.ylabel("")
    plt.tight_layout()
    plt.savefig(path, dpi=220)
    plt.close()

# ============================================================
# 5. Load ETF returns, policy descriptions, and signal weights
# ============================================================

print("\n" + "=" * 80)
print("Step 1: Loading ETF returns and Notebook 08 signal weights")
print("=" * 80)

etf_returns, etf_return_path = load_etf_return_panel()
print("ETF return panel:", etf_return_path)
print("ETF return shape:", etf_returns.shape)
print("ETF date range  :", etf_returns.index.min().date(), "to", etf_returns.index.max().date())

policy_description_candidates = [
    NOTEBOOK08_TABLE_DIR / "policy_descriptions.csv",
    TABLE_DIR / f"table_64_policy_descriptions_stronger_baselines_{NOTEBOOK08_RUN_ID}.csv",
]

policy_description_path = find_first_existing(policy_description_candidates)

if policy_description_path is None:
    raise FileNotFoundError(
        "Could not find Notebook 08 policy descriptions. Tried:\n"
        + "\n".join(str(p) for p in policy_description_candidates)
    )

policy_description_df = pd.read_csv(policy_description_path)

required_cols = ["policy_name", "policy_type"]
missing = [c for c in required_cols if c not in policy_description_df.columns]
if missing:
    raise ValueError(f"Policy description file missing columns: {missing}")

print("Policy description path:", policy_description_path)
print("Policy descriptions    :", policy_description_df.shape)

policy_weight_dict = {}
policy_type_dict = {}

for _, row in policy_description_df.iterrows():
    policy_name = row["policy_name"]
    policy_type = row["policy_type"]

    path = NOTEBOOK08_WEIGHT_DIR / f"signal_weights_{safe_name(policy_name)}.parquet"

    if not path.exists():
        alt = NOTEBOOK08_WEIGHT_DIR / f"signal_weights_{safe_name(policy_name)}.csv"
        if alt.exists():
            path = alt
        else:
            print(f"WARNING: missing signal weight file for {policy_name}")
            continue

    w = read_table_auto(path)
    w = w.reindex(columns=ALL_ASSETS).fillna(0.0)
    w = normalize_rows(w)

    policy_weight_dict[policy_name] = w
    policy_type_dict[policy_name] = policy_type

print("Loaded policy weights:", len(policy_weight_dict))
print(pd.Series(policy_type_dict).value_counts().to_string())

if not policy_weight_dict:
    raise ValueError("No policy weights were loaded.")

# ============================================================
# 6. Load Notebook 07 probability dates for strict test alignment
# ============================================================

print("\n" + "=" * 80)
print("Step 2: Loading Notebook 07 probability dates")
print("=" * 80)

input_index_df = pd.read_csv(NOTEBOOK08_INPUT_INDEX)

p20_path = None
p60_path = None

for _, row in input_index_df.iterrows():
    target_col = row["target_col"]

    path = Path(row["probability_path_parquet"])
    if not path.exists():
        path = Path(row["probability_path_csv"])

    if "20d" in target_col:
        p20_path = path
    elif "60d" in target_col:
        p60_path = path

if p20_path is None or p60_path is None:
    raise ValueError("Could not locate both 20d and 60d probability files from Notebook 07 index.")

p20_raw = read_table_auto(p20_path)
p60_raw = read_table_auto(p60_path)

p20_oos = latest_fold_deduplicate(p20_raw, split_filter=["validation", "test"])
p60_oos = latest_fold_deduplicate(p60_raw, split_filter=["validation", "test"])

p20_test = latest_fold_deduplicate(p20_raw, split_filter=["test"])
p60_test = latest_fold_deduplicate(p60_raw, split_filter=["test"])

prob_oos_dates = p20_oos.index.intersection(p60_oos.index).intersection(etf_returns.index)
prob_test_dates = p20_test.index.intersection(p60_test.index).intersection(etf_returns.index)

print("20d OOS probabilities :", p20_oos.shape, p20_oos.index.min().date(), "to", p20_oos.index.max().date())
print("60d OOS probabilities :", p60_oos.shape, p60_oos.index.min().date(), "to", p60_oos.index.max().date())
print("Common prob OOS dates :", len(prob_oos_dates), prob_oos_dates.min().date(), "to", prob_oos_dates.max().date())
print("Common prob test dates:", len(prob_test_dates), prob_test_dates.min().date(), "to", prob_test_dates.max().date())

# ============================================================
# 7. Build aligned OOS and aligned test indexes
# ============================================================

print("\n" + "=" * 80)
print("Step 3: Creating aligned evaluation date indexes")
print("=" * 80)

# Align all policies to dates available for AURORA purged probabilities.
# This fixes the 508 vs 545 day mismatch from Notebook 08.
common_policy_dates = prob_oos_dates.copy()

for policy_name, w in policy_weight_dict.items():
    common_policy_dates = common_policy_dates.intersection(w.index)

common_policy_dates = common_policy_dates.intersection(etf_returns.index).sort_values()

aligned_oos_dates = pd.DatetimeIndex(common_policy_dates).sort_values()
aligned_test_dates = aligned_oos_dates.intersection(prob_test_dates).sort_values()

if len(aligned_oos_dates) == 0:
    raise ValueError("Aligned OOS date index is empty.")

if len(aligned_test_dates) == 0:
    raise ValueError("Aligned strict test date index is empty.")

alignment_report = {
    "run_id": RUN_ID,
    "notebook08_run_id": NOTEBOOK08_RUN_ID,
    "notebook07_run_id": NOTEBOOK07_RUN_ID,
    "aligned_oos_n_days": int(len(aligned_oos_dates)),
    "aligned_oos_start": str(aligned_oos_dates.min().date()),
    "aligned_oos_end": str(aligned_oos_dates.max().date()),
    "aligned_test_n_days": int(len(aligned_test_dates)),
    "aligned_test_start": str(aligned_test_dates.min().date()),
    "aligned_test_end": str(aligned_test_dates.max().date()),
    "n_policies": int(len(policy_weight_dict)),
    "alignment_rule": "Intersection of all policy signal-weight dates, ETF return dates, and common 20d/60d purged probability dates.",
}

alignment_report_df = pd.DataFrame([alignment_report])
alignment_report_df.to_csv(TABLE_RUN_DIR / "aligned_evaluation_date_report.csv", index=False)
alignment_report_df.to_csv(TABLE_DIR / f"table_66_aligned_evaluation_date_report_{RUN_ID}.csv", index=False)

print("Aligned OOS dates :", len(aligned_oos_dates), aligned_oos_dates.min().date(), "to", aligned_oos_dates.max().date())
print("Aligned test dates:", len(aligned_test_dates), aligned_test_dates.min().date(), "to", aligned_test_dates.max().date())
print("All policies will now be evaluated on identical date counts.")

# ============================================================
# 8. Backtest every policy on aligned OOS and aligned test dates
# ============================================================

print("\n" + "=" * 80)
print("Step 4: Backtesting all policies on aligned dates")
print("=" * 80)

all_oos_return_frames = []
all_test_return_frames = []
all_oos_weight_frames = []
all_test_weight_frames = []

oos_metric_rows = []
test_metric_rows = []

for policy_name, signal_w in policy_weight_dict.items():
    policy_type = policy_type_dict[policy_name]

    print("Backtesting aligned:", policy_name)

    oos_returns, oos_weights = backtest_on_fixed_index(
        policy_name=policy_name,
        signal_weights=signal_w,
        etf_returns=etf_returns,
        evaluation_index=aligned_oos_dates,
    )

    test_returns, test_weights = backtest_on_fixed_index(
        policy_name=policy_name,
        signal_weights=signal_w,
        etf_returns=etf_returns,
        evaluation_index=aligned_test_dates,
    )

    oos_metrics = performance_metrics(oos_returns)
    oos_metrics["run_id"] = RUN_ID
    oos_metrics["policy_name"] = policy_name
    oos_metrics["policy_type"] = policy_type
    oos_metrics["period"] = "aligned_oos_validation_and_test"
    oos_metrics["rebalance_frequency"] = REBALANCE_FREQUENCY
    oos_metrics["transaction_cost_rate"] = TRANSACTION_COST_RATE
    oos_metric_rows.append(oos_metrics)

    test_metrics = performance_metrics(test_returns)
    test_metrics["run_id"] = RUN_ID
    test_metrics["policy_name"] = policy_name
    test_metrics["policy_type"] = policy_type
    test_metrics["period"] = "aligned_strict_test_only"
    test_metrics["rebalance_frequency"] = REBALANCE_FREQUENCY
    test_metrics["transaction_cost_rate"] = TRANSACTION_COST_RATE
    test_metric_rows.append(test_metrics)

    all_oos_return_frames.append(oos_returns)
    all_test_return_frames.append(test_returns)

    oos_w = oos_weights.copy()
    oos_w.index.name = "date"
    oos_w.insert(0, "policy_name", policy_name)
    all_oos_weight_frames.append(oos_w)

    test_w = test_weights.copy()
    test_w.index.name = "date"
    test_w.insert(0, "policy_name", policy_name)
    all_test_weight_frames.append(test_w)

    oos_returns.to_parquet(RETURN_DIR / f"aligned_oos_returns_{safe_name(policy_name)}.parquet")
    oos_returns.to_csv(RETURN_DIR / f"aligned_oos_returns_{safe_name(policy_name)}.csv")

    test_returns.to_parquet(RETURN_DIR / f"aligned_test_returns_{safe_name(policy_name)}.parquet")
    test_returns.to_csv(RETURN_DIR / f"aligned_test_returns_{safe_name(policy_name)}.csv")

    oos_weights.to_parquet(WEIGHT_DIR / f"aligned_oos_daily_weights_{safe_name(policy_name)}.parquet")
    oos_weights.to_csv(WEIGHT_DIR / f"aligned_oos_daily_weights_{safe_name(policy_name)}.csv")

    test_weights.to_parquet(WEIGHT_DIR / f"aligned_test_daily_weights_{safe_name(policy_name)}.parquet")
    test_weights.to_csv(WEIGHT_DIR / f"aligned_test_daily_weights_{safe_name(policy_name)}.csv")

all_oos_returns_df = pd.concat(all_oos_return_frames, axis=0)
all_test_returns_df = pd.concat(all_test_return_frames, axis=0)

all_oos_weights_df = pd.concat(all_oos_weight_frames, axis=0)
all_test_weights_df = pd.concat(all_test_weight_frames, axis=0)

oos_performance_df = pd.DataFrame(oos_metric_rows)
test_performance_df = pd.DataFrame(test_metric_rows)

# Verify exact equal n_days.
assert oos_performance_df["n_days"].nunique() == 1, "OOS n_days mismatch remains."
assert test_performance_df["n_days"].nunique() == 1, "Test n_days mismatch remains."

all_oos_returns_df.to_parquet(RETURN_DIR / "aligned_oos_returns_all_policies.parquet")
all_oos_returns_df.to_csv(RETURN_DIR / "aligned_oos_returns_all_policies.csv")

all_test_returns_df.to_parquet(RETURN_DIR / "aligned_test_returns_all_policies.parquet")
all_test_returns_df.to_csv(RETURN_DIR / "aligned_test_returns_all_policies.csv")

all_oos_weights_df.to_parquet(WEIGHT_DIR / "aligned_oos_daily_weights_all_policies.parquet")
all_oos_weights_df.to_csv(WEIGHT_DIR / "aligned_oos_daily_weights_all_policies.csv")

all_test_weights_df.to_parquet(WEIGHT_DIR / "aligned_test_daily_weights_all_policies.parquet")
all_test_weights_df.to_csv(WEIGHT_DIR / "aligned_test_daily_weights_all_policies.csv")

oos_performance_df.to_csv(TABLE_RUN_DIR / "aligned_oos_performance.csv", index=False)
oos_performance_df.to_csv(TABLE_DIR / f"table_67_aligned_oos_performance_{RUN_ID}.csv", index=False)

test_performance_df.to_csv(TABLE_RUN_DIR / "aligned_test_only_performance.csv", index=False)
test_performance_df.to_csv(TABLE_DIR / f"table_68_aligned_test_only_performance_{RUN_ID}.csv", index=False)

print("\nAligned OOS performance n_days:", oos_performance_df["n_days"].unique())
print("Aligned test performance n_days:", test_performance_df["n_days"].unique())

# ============================================================
# 9. Rankings and AURORA-vs-benchmark comparisons
# ============================================================

print("\n" + "=" * 80)
print("Step 5: Creating aligned rankings and comparisons")
print("=" * 80)

rank_oos_df = add_composite_rank(oos_performance_df)
rank_test_df = add_composite_rank(test_performance_df)

rank_oos_df.to_csv(TABLE_RUN_DIR / "aligned_oos_rankings.csv", index=False)
rank_oos_df.to_csv(TABLE_DIR / f"table_69_aligned_oos_rankings_{RUN_ID}.csv", index=False)

rank_test_df.to_csv(TABLE_RUN_DIR / "aligned_test_only_rankings.csv", index=False)
rank_test_df.to_csv(TABLE_DIR / f"table_70_aligned_test_only_rankings_{RUN_ID}.csv", index=False)

def make_aurora_vs_best_benchmark(rank_df, period_label):
    benchmark_df = rank_df[rank_df["policy_type"].isin(BENCHMARK_TYPES)].copy()
    aurora_df = rank_df[rank_df["policy_type"].astype(str).str.startswith("AURORA")].copy()

    if benchmark_df.empty:
        raise ValueError(f"No benchmark policies found for {period_label}.")

    if aurora_df.empty:
        raise ValueError(f"No AURORA policies found for {period_label}.")

    best_benchmark = benchmark_df.sort_values("allocation_composite_rank").iloc[0]
    best_overall = rank_df.sort_values("allocation_composite_rank").iloc[0]
    best_aurora = aurora_df.sort_values("allocation_composite_rank").iloc[0]

    rows = []

    for _, row in aurora_df.sort_values("allocation_composite_rank").iterrows():
        rows.append({
            "period": period_label,
            "aurora_policy": row["policy_name"],
            "benchmark_policy": best_benchmark["policy_name"],
            "aurora_total_return": row["total_return"],
            "benchmark_total_return": best_benchmark["total_return"],
            "excess_total_return": row["total_return"] - best_benchmark["total_return"],
            "aurora_annual_return": row["annual_return"],
            "benchmark_annual_return": best_benchmark["annual_return"],
            "excess_annual_return": row["annual_return"] - best_benchmark["annual_return"],
            "aurora_sharpe": row["sharpe_ratio"],
            "benchmark_sharpe": best_benchmark["sharpe_ratio"],
            "excess_sharpe": row["sharpe_ratio"] - best_benchmark["sharpe_ratio"],
            "aurora_sortino": row["sortino_ratio"],
            "benchmark_sortino": best_benchmark["sortino_ratio"],
            "excess_sortino": row["sortino_ratio"] - best_benchmark["sortino_ratio"],
            "aurora_max_drawdown": row["max_drawdown"],
            "benchmark_max_drawdown": best_benchmark["max_drawdown"],
            "drawdown_improvement": row["max_drawdown"] - best_benchmark["max_drawdown"],
            "aurora_composite_rank": row["allocation_composite_rank"],
            "benchmark_composite_rank": best_benchmark["allocation_composite_rank"],
            "composite_rank_improvement": best_benchmark["allocation_composite_rank"] - row["allocation_composite_rank"],
        })

    return pd.DataFrame(rows), best_aurora, best_benchmark, best_overall

aurora_vs_oos_df, best_aurora_oos, best_benchmark_oos, best_overall_oos = make_aurora_vs_best_benchmark(
    rank_oos_df,
    "aligned_oos_validation_and_test",
)

aurora_vs_test_df, best_aurora_test, best_benchmark_test, best_overall_test = make_aurora_vs_best_benchmark(
    rank_test_df,
    "aligned_strict_test_only",
)

aurora_vs_benchmark_df = pd.concat([aurora_vs_oos_df, aurora_vs_test_df], ignore_index=True)

aurora_vs_benchmark_df.to_csv(TABLE_RUN_DIR / "aligned_aurora_vs_best_benchmark.csv", index=False)
aurora_vs_benchmark_df.to_csv(TABLE_DIR / f"table_71_aligned_aurora_vs_best_benchmark_{RUN_ID}.csv", index=False)

print("\nAligned OOS top rankings:")
print(rank_oos_df[[
    "policy_name",
    "policy_type",
    "n_days",
    "total_return",
    "annual_return",
    "annual_volatility",
    "sharpe_ratio",
    "sortino_ratio",
    "max_drawdown",
    "calmar_ratio",
    "allocation_composite_rank",
]].head(25).to_string(index=False))

print("\nAligned strict test top rankings:")
print(rank_test_df[[
    "policy_name",
    "policy_type",
    "n_days",
    "total_return",
    "annual_return",
    "annual_volatility",
    "sharpe_ratio",
    "sortino_ratio",
    "max_drawdown",
    "calmar_ratio",
    "allocation_composite_rank",
]].head(25).to_string(index=False))

print("\nAligned AURORA vs best benchmark:")
print(aurora_vs_benchmark_df.to_string(index=False))

# ============================================================
# 10. Compare Notebook 08 original vs Notebook 08B aligned
# ============================================================

print("\n" + "=" * 80)
print("Step 6: Comparing original Notebook 08 with aligned Notebook 08B")
print("=" * 80)

original_oos_candidates = [
    NOTEBOOK08_TABLE_DIR / "allocation_rankings_oos_validation_and_test.csv",
    TABLE_DIR / f"table_61_allocation_rankings_oos_{NOTEBOOK08_RUN_ID}.csv",
]

original_test_candidates = [
    NOTEBOOK08_TABLE_DIR / "allocation_rankings_strict_test_only.csv",
    TABLE_DIR / f"table_62_allocation_rankings_test_only_{NOTEBOOK08_RUN_ID}.csv",
]

original_oos_path = find_first_existing(original_oos_candidates)
original_test_path = find_first_existing(original_test_candidates)

comparison_rows = []

if original_oos_path is not None:
    original_oos_df = pd.read_csv(original_oos_path)

    for policy in sorted(set(original_oos_df["policy_name"]).intersection(rank_oos_df["policy_name"])):
        old = original_oos_df[original_oos_df["policy_name"] == policy].iloc[0]
        new = rank_oos_df[rank_oos_df["policy_name"] == policy].iloc[0]

        comparison_rows.append({
            "period": "oos",
            "policy_name": policy,
            "old_n_days": old["n_days"],
            "new_n_days": new["n_days"],
            "old_total_return": old["total_return"],
            "new_total_return": new["total_return"],
            "delta_total_return": new["total_return"] - old["total_return"],
            "old_sharpe": old["sharpe_ratio"],
            "new_sharpe": new["sharpe_ratio"],
            "delta_sharpe": new["sharpe_ratio"] - old["sharpe_ratio"],
            "old_max_drawdown": old["max_drawdown"],
            "new_max_drawdown": new["max_drawdown"],
            "delta_max_drawdown": new["max_drawdown"] - old["max_drawdown"],
        })

if original_test_path is not None:
    original_test_df = pd.read_csv(original_test_path)

    for policy in sorted(set(original_test_df["policy_name"]).intersection(rank_test_df["policy_name"])):
        old = original_test_df[original_test_df["policy_name"] == policy].iloc[0]
        new = rank_test_df[rank_test_df["policy_name"] == policy].iloc[0]

        comparison_rows.append({
            "period": "strict_test",
            "policy_name": policy,
            "old_n_days": old["n_days"],
            "new_n_days": new["n_days"],
            "old_total_return": old["total_return"],
            "new_total_return": new["total_return"],
            "delta_total_return": new["total_return"] - old["total_return"],
            "old_sharpe": old["sharpe_ratio"],
            "new_sharpe": new["sharpe_ratio"],
            "delta_sharpe": new["sharpe_ratio"] - old["sharpe_ratio"],
            "old_max_drawdown": old["max_drawdown"],
            "new_max_drawdown": new["max_drawdown"],
            "delta_max_drawdown": new["max_drawdown"] - old["max_drawdown"],
        })

comparison_df = pd.DataFrame(comparison_rows)

comparison_df.to_csv(TABLE_RUN_DIR / "notebook08_vs_08B_alignment_comparison.csv", index=False)
comparison_df.to_csv(TABLE_DIR / f"table_72_notebook08_vs_08B_alignment_comparison_{RUN_ID}.csv", index=False)

print("\nOriginal-vs-aligned comparison preview:")
print(comparison_df.head(30).to_string(index=False))

# ============================================================
# 11. Diagnostic summary
# ============================================================

print("\n" + "=" * 80)
print("Step 7: Creating aligned diagnostic summary")
print("=" * 80)

diagnostic_rows = []

diagnostic_rows.append({
    "question": "What does OOS mean?",
    "finding": "Out-of-sample",
    "evidence": (
        "In this project, OOS means evaluation dates not used for fitting the model. "
        "Here, aligned OOS combines purged walk-forward validation and test probability dates."
    ),
})

diagnostic_rows.append({
    "question": "Was the Notebook 08 OOS date-count mismatch fixed?",
    "finding": "Yes",
    "evidence": (
        f"All policies now use {len(aligned_oos_dates)} aligned OOS days "
        f"from {aligned_oos_dates.min().date()} to {aligned_oos_dates.max().date()}."
    ),
})

diagnostic_rows.append({
    "question": "Was strict test-only comparison aligned?",
    "finding": "Yes",
    "evidence": (
        f"All policies now use {len(aligned_test_dates)} aligned strict test days "
        f"from {aligned_test_dates.min().date()} to {aligned_test_dates.max().date()}."
    ),
})

diagnostic_rows.append({
    "question": "Does best AURORA beat best benchmark in aligned OOS composite rank?",
    "finding": "Yes" if best_aurora_oos["allocation_composite_rank"] < best_benchmark_oos["allocation_composite_rank"] else "No",
    "evidence": (
        f"Best AURORA={best_aurora_oos['policy_name']}, "
        f"rank={best_aurora_oos['allocation_composite_rank']:.2f}, "
        f"Sharpe={best_aurora_oos['sharpe_ratio']:.4f}; "
        f"Best benchmark={best_benchmark_oos['policy_name']}, "
        f"rank={best_benchmark_oos['allocation_composite_rank']:.2f}, "
        f"Sharpe={best_benchmark_oos['sharpe_ratio']:.4f}."
    ),
})

diagnostic_rows.append({
    "question": "Which policy is best overall in aligned OOS composite rank?",
    "finding": str(best_overall_oos["policy_name"]),
    "evidence": (
        f"Policy type={best_overall_oos['policy_type']}, "
        f"total_return={best_overall_oos['total_return']:.4f}, "
        f"Sharpe={best_overall_oos['sharpe_ratio']:.4f}, "
        f"max_drawdown={best_overall_oos['max_drawdown']:.4f}."
    ),
})

diagnostic_rows.append({
    "question": "Does best AURORA beat best benchmark in aligned strict test composite rank?",
    "finding": "Yes" if best_aurora_test["allocation_composite_rank"] < best_benchmark_test["allocation_composite_rank"] else "No",
    "evidence": (
        f"Best AURORA={best_aurora_test['policy_name']}, "
        f"rank={best_aurora_test['allocation_composite_rank']:.2f}, "
        f"Sharpe={best_aurora_test['sharpe_ratio']:.4f}; "
        f"Best benchmark={best_benchmark_test['policy_name']}, "
        f"rank={best_benchmark_test['allocation_composite_rank']:.2f}, "
        f"Sharpe={best_benchmark_test['sharpe_ratio']:.4f}."
    ),
})

diagnostic_df = pd.DataFrame(diagnostic_rows)
diagnostic_df.to_csv(TABLE_RUN_DIR / "aligned_diagnostic_summary.csv", index=False)
diagnostic_df.to_csv(TABLE_DIR / f"table_73_aligned_diagnostic_summary_{RUN_ID}.csv", index=False)

print(diagnostic_df.to_string(index=False))

# ============================================================
# 12. Plots and paper figures
# ============================================================

print("\n" + "=" * 80)
print("Step 8: Creating aligned plots and paper figures")
print("=" * 80)

top_oos_policies = rank_oos_df.head(12)["policy_name"].tolist()
top_test_policies = rank_test_df.head(12)["policy_name"].tolist()

plot_equity_curves(
    all_oos_returns_df,
    path=PLOT_DIR / "aligned_oos_equity_curves_top12.png",
    title="Aligned OOS equity curves: AURORA vs stronger baselines",
    policies=top_oos_policies,
)

plot_drawdowns(
    all_oos_returns_df,
    path=PLOT_DIR / "aligned_oos_drawdowns_top12.png",
    title="Aligned OOS drawdowns: AURORA vs stronger baselines",
    policies=top_oos_policies,
)

plot_metric_bar(
    rank_oos_df,
    metric="sharpe_ratio",
    path=PLOT_DIR / "aligned_oos_sharpe_by_policy.png",
    title="Aligned OOS Sharpe ratio by policy",
    higher_is_better=True,
    top_n=25,
)

plot_metric_bar(
    rank_oos_df,
    metric="total_return",
    path=PLOT_DIR / "aligned_oos_total_return_by_policy.png",
    title="Aligned OOS total return by policy",
    higher_is_better=True,
    top_n=25,
)

plot_metric_bar(
    rank_oos_df,
    metric="max_drawdown",
    path=PLOT_DIR / "aligned_oos_max_drawdown_by_policy.png",
    title="Aligned OOS maximum drawdown by policy",
    higher_is_better=True,
    top_n=25,
)

plot_equity_curves(
    all_test_returns_df,
    path=PLOT_DIR / "aligned_test_equity_curves_top12.png",
    title="Aligned strict test equity curves: AURORA vs stronger baselines",
    policies=top_test_policies,
)

plot_drawdowns(
    all_test_returns_df,
    path=PLOT_DIR / "aligned_test_drawdowns_top12.png",
    title="Aligned strict test drawdowns: AURORA vs stronger baselines",
    policies=top_test_policies,
)

plot_metric_bar(
    rank_test_df,
    metric="sharpe_ratio",
    path=PLOT_DIR / "aligned_test_sharpe_by_policy.png",
    title="Aligned strict test Sharpe ratio by policy",
    higher_is_better=True,
    top_n=25,
)

plot_metric_bar(
    rank_test_df,
    metric="total_return",
    path=PLOT_DIR / "aligned_test_total_return_by_policy.png",
    title="Aligned strict test total return by policy",
    higher_is_better=True,
    top_n=25,
)

plot_metric_bar(
    rank_test_df,
    metric="max_drawdown",
    path=PLOT_DIR / "aligned_test_max_drawdown_by_policy.png",
    title="Aligned strict test maximum drawdown by policy",
    higher_is_better=True,
    top_n=25,
)

paper_figure_files = [
    "aligned_oos_equity_curves_top12.png",
    "aligned_oos_drawdowns_top12.png",
    "aligned_oos_sharpe_by_policy.png",
    "aligned_oos_total_return_by_policy.png",
    "aligned_oos_max_drawdown_by_policy.png",
    "aligned_test_equity_curves_top12.png",
    "aligned_test_drawdowns_top12.png",
    "aligned_test_sharpe_by_policy.png",
    "aligned_test_total_return_by_policy.png",
    "aligned_test_max_drawdown_by_policy.png",
]

for fname in paper_figure_files:
    src = PLOT_DIR / fname
    if src.exists():
        dst = PAPER_FIGURE_DIR / fname
        dst.write_bytes(src.read_bytes())

        global_dst = FIGURE_DIR / f"{Path(fname).stem}_{RUN_ID}.png"
        global_dst.write_bytes(src.read_bytes())

print("Plots saved to        :", PLOT_DIR)
print("Paper figures saved to:", PAPER_FIGURE_DIR)

# ============================================================
# 13. Validation report and manifest
# ============================================================

print("\n" + "=" * 80)
print("Step 9: Saving validation report and manifest")
print("=" * 80)

validation_report = {
    "project_code": PROJECT_CODE,
    "notebook": "08B_AURORA_aligned_oos_reanalysis.ipynb",
    "run_timestamp_utc": RUN_TIMESTAMP,
    "run_id": RUN_ID,
    "notebook07_run_id": NOTEBOOK07_RUN_ID,
    "notebook08_run_id": NOTEBOOK08_RUN_ID,
    "notebook07_root": str(NOTEBOOK07_ROOT),
    "notebook08_root": str(NOTEBOOK08_ROOT),
    "notebook08_input_index": str(NOTEBOOK08_INPUT_INDEX),
    "etf_return_panel": str(etf_return_path),
    "transaction_cost_rate": TRANSACTION_COST_RATE,
    "rebalance_frequency": REBALANCE_FREQUENCY,
    "alignment_report": alignment_report,
    "n_policies": int(len(policy_weight_dict)),
    "policy_type_counts": pd.Series(policy_type_dict).value_counts().to_dict(),
    "best_aurora_oos": best_aurora_oos.to_dict(),
    "best_benchmark_oos": best_benchmark_oos.to_dict(),
    "best_overall_oos": best_overall_oos.to_dict(),
    "best_aurora_test": best_aurora_test.to_dict(),
    "best_benchmark_test": best_benchmark_test.to_dict(),
    "best_overall_test": best_overall_test.to_dict(),
    "diagnostics": diagnostic_df.to_dict(orient="records"),
    "educational_note": (
        "This notebook performs aligned research backtests only and does not provide personalized financial advice."
    ),
    "output_paths": {
        "run_root": str(RUN_ROOT),
        "tables": str(TABLE_RUN_DIR),
        "returns": str(RETURN_DIR),
        "weights": str(WEIGHT_DIR),
        "plots": str(PLOT_DIR),
        "paper_figures": str(PAPER_FIGURE_DIR),
    },
}

validation_report_path = REPORT_RUN_DIR / "AURORA_08B_aligned_oos_validation_report.json"
validation_report_global_path = REPORT_DIR / f"AURORA_08B_aligned_oos_validation_report_{RUN_ID}.json"

save_json(validation_report_path, validation_report)
save_json(validation_report_global_path, validation_report)

manifest_df = make_file_manifest(RUN_ROOT)

manifest_path = REPORT_RUN_DIR / "AURORA_08B_file_manifest_SHA256.csv"
manifest_global_path = REPORT_DIR / f"AURORA_08B_file_manifest_SHA256_{RUN_ID}.csv"

manifest_df.to_csv(manifest_path, index=False)
manifest_df.to_csv(manifest_global_path, index=False)

# ============================================================
# 14. Final summary
# ============================================================

print("\n" + "=" * 80)
print("AURORA-TWETF NOTEBOOK 08B COMPLETE")
print("=" * 80)
print("Run ID                              :", RUN_ID)
print("Run root                            :", RUN_ROOT)
print("Aligned date report                 :", TABLE_DIR / f"table_66_aligned_evaluation_date_report_{RUN_ID}.csv")
print("Aligned OOS performance             :", TABLE_DIR / f"table_67_aligned_oos_performance_{RUN_ID}.csv")
print("Aligned test-only performance       :", TABLE_DIR / f"table_68_aligned_test_only_performance_{RUN_ID}.csv")
print("Aligned OOS rankings                :", TABLE_DIR / f"table_69_aligned_oos_rankings_{RUN_ID}.csv")
print("Aligned test-only rankings          :", TABLE_DIR / f"table_70_aligned_test_only_rankings_{RUN_ID}.csv")
print("Aligned AURORA vs benchmark         :", TABLE_DIR / f"table_71_aligned_aurora_vs_best_benchmark_{RUN_ID}.csv")
print("Notebook 08 vs 08B comparison       :", TABLE_DIR / f"table_72_notebook08_vs_08B_alignment_comparison_{RUN_ID}.csv")
print("Aligned diagnostic summary          :", TABLE_DIR / f"table_73_aligned_diagnostic_summary_{RUN_ID}.csv")
print("Returns directory                   :", RETURN_DIR)
print("Weights directory                   :", WEIGHT_DIR)
print("Plots directory                     :", PLOT_DIR)
print("Paper figures directory             :", PAPER_FIGURE_DIR)
print("Validation report                   :", validation_report_path)
print("Manifest                            :", manifest_path)
print("=" * 80)

print("\nRecommended next notebook:")
print("09_AURORA_validation_optimized_regime_templates.ipynb")

Mounted at /content/drive
AURORA-TWETF Notebook 08B: Aligned OOS Reanalysis
Timestamp UTC       : 2026-06-24T07:08:27Z
Run ID              : 20260624_070827
Notebook 07 root    : /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/purged_walk_forward_models/run_20260624_031817
Notebook 08 root    : /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/stronger_financial_baselines_purged_allocation/run_20260624_034204
Notebook 08 registry: /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/purged_walk_forward_models/run_20260624_031817/NOTEBOOK08_OR_ALLOCATION_INPUT_INDEX_PURGED_WF.csv
Run root            : /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/aligned_oos_reanalysis/run_20260624_070827

Step 1: Loading ETF returns and Notebook 08 signal weights
ETF return panel: /content/drive/MyDrive/AURORA_TWETF/data/panels/AURORA_etf_return_panel.parquet
ETF return shape: (1426, 4)
ETF date range  : 2021-01-01 to 2026-06-23
Policy description path: /content/drive/MyD